In [55]:
import pandas as pd

In [56]:
df = pd.read_csv("./commit_info_2025.csv")

C:\Users\nopes\AppData\Local\Temp\ipykernel_9248\1833696204.py:1: DtypeWarning: Columns (2,4,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,2

In [57]:
df = df.drop(0) #get rid of numbers to make sure csv was read correctly
print(df.shape)
df

(6198, 396)


,ID,Project_Name,Author_ID,Created,Nr_Files,Tracking_ID,files,Commit Message;;;;;,Unnamed: 8,Unnamed: 9,...,Unnamed: 386,Unnamed: 387,Unnamed: 388,Unnamed: 389,Unnamed: 390,Unnamed: 391,Unnamed: 392,Unnamed: 393,Unnamed: 394,Unnamed: 395
1,I0016c57575eda0c77b49ae577d2a322103d80d03,cat/tapas/cfg/rmca,1069329,41:01.0,1,JIRA-ID: BSS_CHA-23938,['config/host-config.json'],Add new network to RMCA WM cluster 1 Accident...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"I001768f8d53c879f8c826e98d24b0130eef344f1,rmca...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,I0052410028ad8ea53a8b2e4e2e8584391e03927a,rmca/com.ericsson.bss.rmca.cus.packaging.msv,9381,37:55.0,1,SCM,['bom/pom.xml'],Set snapshot version RMCA CUS - 2.80.3-SNAPSH...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,I0056c24118b756ee9463dfbbe69f5ddfbeb02c05,charging/com.ericsson.bss.rm.coba.gui,9255,12:25.0,1,TEST,['ui/src/coba/components/globallist-specificat...,Fix invalid HTML Incorrect matching parenthes...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,"I006526646bde76663d20e9944cb85c0c0307b4fc,rmca...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6194,"Iffcac18d598dc694e19b0e0d3933747a5ae816a4,rmca...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6195,"Iffd80a804135253ae45cd2f8865491f57dd547c2,rmca...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6196,"Iffe7fae3598358e3b9bdb740a972f154f638609b,rmca...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6197,"Ifff275a1012ba92bbbb00e6214c64d6f9fbc30a1,rmca...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [457]:
def handle_normal_row(row):
    pr = row[0]
    author = row[2]
    file = clean_file(row[6])
    title = clean_title(row[7])
    body = 'nan'
    
    if (len(file) == 0):
        return []
    else:
        return [pr, ';', file, ';', title, ';', body, ';', author]

In [458]:
def handle_single_broken(rowStr): #broken instance on one line. Ends in ';;;;;'
    rowlist = rowStr.split(',')
    length = len(rowlist)
    
    pr = rowlist[0]
    author = rowlist[2]
    
    files = ""
    for j in range(6, length-1):
        file = clean_file(rowlist[j])
        if(len(file) != 0):
            files += file
            
            if(j != length-2):
                files += ';'
                
    title = clean_title(rowlist[length-1])
    body = 'nan'
    
    if(len(files) == 0):
        return []
    else:
        return [pr, ';', files, ';', title, ';', body, ';', author]
    

In [459]:
pr = ""
hold_files = ""
partial_file = ""
title = ""
body = ""
author = ""


def handle_multiple_broken_start(row): #broken instance on multiple lines; Doesn't end in ';;;;;', last line does
    global pr
    global hold_files
    global partial_file
    global title
    global body
    global author
    
    pr = ""
    hold_files = ""
    partial_file = ""
    title = ""
    body = ""
    author = ""
    
    rowlist = row.split(',')
    length = len(rowlist)
    
    pr = rowlist[0]
    author = rowlist[2]
    
    for j in range(6, length-1):
        file = clean_file(rowlist[j])
        if(len(file) != 0):
            hold_files += file
            hold_files += ';'
    
    #add in broken ending file
    partial_file = rowlist[length-1].replace(';','')
    
def handle_extra_files(row): #extra files following
    global partial_file
    global hold_files
    
    length = pd.DataFrame(row).count().iloc[0]
    
    #handle second half of broken ending file
    partial_file += row[0]
    partial_file = clean_file(partial_file)
    if(len(partial_file) != 0):
        hold_files += partial_file
        hold_files += ';'
    
    for j in range(1, length-1):
        file = clean_file(row[j])
        if(len(file) != 0):
            hold_files += file
            hold_files += ';'
            
    #handle broken ending file at the end of this row
    partial_file = row[length-1].replace(';','')
    
def handle_multiple_broken_end(row): #broken instance on multiple lines. Ends in ';;;;;'
    rowlist = row.split(',')
    length = len(rowlist)
    
    global partial_file
    global hold_files
    
    partial_file += rowlist[0]
    partial_file = clean_file(partial_file)
    if(len(partial_file) != 0):
        hold_files += partial_file
        if(length != 2):
            hold_files += ';'
    
    for j in range(1,length-1):
        file = clean_file(rowlist[j])
        if(len(file) != 0):
            hold_files += file
            if j != length-2:
                hold_files += ';'
    
    title = clean_title(rowlist[length-1])
    body = 'nan'
    
    global pr
    global author
    
    if(len(hold_files) == 0):
        return []
    else:
        return [pr, ';', hold_files, ';', title, ';', body, ';' , author]
    

In [460]:
def clean_file(file):
    if(type(file) == float): #file == nan => 0 files
        return ""
    if(file.find('.java') == -1):
        return ""
    if(not "/src" in file):
        return ""
    
    file = file.replace("[", "")
    file = file.replace("]", "")
    file = file.replace("\"", "")
    file = file.replace("\'", "")
    file = file.replace(" ", "")
    
    file = file.split('/src',1)[1]
    file = 'src' + file
    return file

def clean_title(title):
    title = title.replace("\n", "")
    title = title.replace("\"", "")
    title = title.replace("\'", "")
    title = title.replace("`", "")
    title = title.replace(";", "")
    title = title.replace(",", "")
    return title

In [461]:
def create_filesPR3BodyTitle2(commitDF):
    filesPR3BodyTitle2List = []
    last_finished = True
    i = 1
    
    for row in commitDF.itertuples(index = False):
        print(i)
        length = pd.DataFrame(row).count().iloc[0]
        
        if(length == 1):
            if(row[0] != ";;;;;"):
                if(row[0].endswith(";;;;;") or row[0].endswith(";;;;")):
                    if(last_finished):
                        list = handle_single_broken(row[0])
                        last_finished = True
                        
                        if not len(list) == 0:
                            print("Added: ", list)
                            filesPR3BodyTitle2List.append(list)
                        
                    else:
                        print("end")
                        list = handle_multiple_broken_end(row[0])
                        last_finished = True
                        
                        if not len(list) == 0:
                            print("Added: ", list)
                            filesPR3BodyTitle2List.append(list)
                        
                else: #row does not end in ;;;;;
                    if(last_finished):
                        print("start")
                        handle_multiple_broken_start(row[0])
                        last_finished = False
                    
                    
                
        else: #length != 1
            if(row[0].find('.') == -1 and last_finished): #the start of a valid row
                list = handle_normal_row(row)
                last_finished = True
                
                if not len(list) == 0:
                    print("Added: ", list)
                    filesPR3BodyTitle2List.append(list)
                            
            if(not last_finished): #need to count extra files
                print("extra files")
                handle_extra_files(row)
                
        i += 1
    return pd.DataFrame(filesPR3BodyTitle2List)

In [462]:
PBT_df = create_filesPR3BodyTitle2(df)

1
2
Added:  ['I001768f8d53c879f8c826e98d24b0130eef344f1', ';', 'src/main/java/com/ericsson/bss/rmca/entitymodificationviews/validation/ValidationErrorCode.java;src/main/java/com/ericsson/bss/rmca/entitymodificationviews/validation/ValidationErrorFactory.java;src/main/java/com/ericsson/bss/rmca/validation/validation/rules/price/AbstractOptimizedConditionValidationRule.java;src/main/java/com/ericsson/bss/rmca/validation/validation/rules/price/OptimizedPolicyValueValidationRule.java;src/main/java/com/ericsson/bss/rmca/validation/validation/rules/price/OptimizedPopValidationRule.java;src/test/java/com/ericsson/bss/rmca/validation/validation/rules/price/TestOptimizedPolicyValueValidationRule.java;src/test/java/com/ericsson/bss/rmca/validation/validation/rules/price/TestOptimizedPopValidationRule.java', ';', 'Validate condition and action spec references', ';', 'nan', ';', '2591']
3
4
5
Added:  ['I006526646bde76663d20e9944cb85c0c0307b4fc', ';', 'src/main/java/com/ericsson/bss/rmca/restservic

Added:  ['I0569d003584292b545f1cd079fa02e519bbb163d', ';', 'src/main/java/com/ericsson/bss/rmca/copilot/erd/acknowledgement/EntityRequirementAcknowledgement.java;src/main/java/com/ericsson/bss/rmca/restserviceimpl/copilot/internal/RestCopilotImpl.java', ';', 'Fix for creation of entity requirement  Update to rest api for ERDs to only include main entity acknowledgement', ';', 'nan', ';', '68']
107
108
Added:  ['I057c7bd75758243f332a0811c4251c9437e1bedb', ';', 'src/main/java/com/ericsson/bss/rmca/configurationprovisioner/tool/GetTestDataPackagesTool.java;src/main/java/com/ericsson/bss/rmca/configurationprovisioner/tool/ProvisionTestDataPackageTool.java;src/main/java/com/ericsson/bss/rmca/copilot/Tool.java;src/main/java/com/ericsson/bss/rmca/copilot/internal/erd/tool/CreateEntityToolProvider.java;src/main/java/com/ericsson/bss/rmca/copilot/internal/erd/tool/GetJsonSchemaTool.java;src/main/java/com/ericsson/bss/rmca/copilot/internal/erd/tool/NavigateToPathTool.java;src/main/java/com/erics

1505
1506
1507
1508
1509
Added:  ['I3e2c47f89c815f2d9198f332960363bf8b33dac3', ';', 'src/main/java/com/ericsson/bss/rmca/entitymodificationviews/rulebuilder/ActionMetaInfo.java;src/main/java/com/ericsson/bss/rmca/entitymodificationviews/validation/ValidationResult.java;src/main/java/com/ericsson/bss/rmca/restserviceimpl/internal/helpers/RestResponseWrapper.java', ';', 'Stop implementing serializable  To resolve sonar issues.  Tracking-Id: EIP', ';', 'nan', ';', '810']
1510
Added:  ['I3e57f0b8bc2f391849de55ca2deb9fb106ca3e8f', ';', 'src/main/java/com/ericsson/bss/rmca/utilities/entity/dependency/entity/PolicyValueSpecificationDependencyAnalyzer.java;src/main/java/com/ericsson/bss/rmca/utilities/entity/dependency/entity/ProductOfferingDependencyAnalyzer.java;src/main/java/com/ericsson/bss/rmca/utilities/entity/dependency/entity/ProductOfferingPriceSpecificationDependencyAnalyzer.java;', ';', ' POPS and PO so that references from improved price model are reported without path (excluded fr

Added:  ['I70a35734fec6d915567a53782b306d7cb08111e1', ';', 'src/main/java/com/ericsson/bss/rmca/restserviceimpl/catalogintegration/internal/converters/CreatorHelper.java;src/main/java/com/ericsson/bss/rmca/restserviceimpl/catalogintegration/internal/converters/CreatorHelperImpl.java;src/test/java/com/ericsson/bss/rmca/restserviceimpl/catalogintegration/internal/converters/TestCreatorHelperImpl.java', ';', 'Update creator helper in catalog integration  Updatas creator helper to make single relation/characteristic creation helper methods public so they can be used for single relations on POP and APOP', ';', 'nan', ';', '68']
2757
2758
2759
Added:  ['I70bc84f888649043ed187b983795feb656e7ca27', ';', 'src/main/java/com/ericsson/bss/rmca/utilities/rule/action/ActionAttributeTypeHelper.java;src/main/java/com/ericsson/bss/rmca/utilities/rule/condition/ConditionAttributeTypeHelper.java;src/test/java/com/ericsson/bss/rmca/utilities/rule/action/TestActionAttributeTypeHelper.java;src/test/java/com

5216
5217
5218
5219
5220
5221
Added:  ['Id8bd0f12c71a6b54feb6e919e822b7cdb763d32b', ';', 'src/main/java/com/ericsson/bss/rmca/cdac/instancegenerator/EntityConverters.java;src/main/java/com/ericsson/bss/rmca/cdac/instancegenerator/EntityServiceVersionedViewsApiGenerator.java;src/main/java/com/ericsson/bss/rmca/cdac/instancegenerator/EntityServiceViewlInstanceConvertersGenerator.java;src/main/java/com/ericsson/bss/rmca/cdac/instancegenerator/instancetypes/InstanceClass.java;src/main/java/com/ericsson/bss/rmca/cdac/instancegenerator/instancetypes/InstanceField.java;src/main/java/com/ericsson/bss/rmca/cdac/instancegenerator/instancetypes/NameMutator.java;src/main/java/com/ericsson/bss/rmca/cdac/instancegenerator/instancetypes/NameType.java', ';', 'Revert Align InstanceGenerator  This reverts commit 882b496dd59cca64288b5f560c9c0ffcf6aed9aa.  Reason for revert: failure in rmca', ';', 'nan', ';', '810']
5222
5223
Added:  ['Id8c888ec75a201cd1f9e82250a5763243675b900', ';', 'src/main/java/com/er

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [463]:
PBT_df

,0,1,2,3,4,5,6,7,8
0,I001768f8d53c879f8c826e98d24b0130eef344f1,;,src/main/java/com/ericsson/bss/rmca/entitymodi...,;,Validate condition and action spec references,;,nan,;,2591
1,I006526646bde76663d20e9944cb85c0c0307b4fc,;,src/main/java/com/ericsson/bss/rmca/restservic...,;,Remove dependency to write rmca in restservice...,;,nan,;,810
2,I006d7ae89ce769c61bd737c38178b44e924a5549,;,src/main/java/com/ericsson/bss/rmca/eventservi...,;,Change bcce schema version to 1.7.0 This upda...,;,nan,;,2427
3,I00756af13d2cb517bed558f5d8758d2a4e19e5f4,;,src/main/java/com/ericsson/bss/rmca/restservic...,;,Update POP Creator for internal realation Upd...,;,nan,;,4679
4,I007586521e73a61abce884421a5e85b2f7d42c9b,;,src/main/java/com/ericsson/bss/rmca/cn/fabric8...,;,DUMMY Tracking-Id:,;,nan,;,1530
...,...,...,...,...,...,...,...,...,...
2493,Iffc1095a3e5ce6a544dfd87de12d541dad4c1e58,;,src/main/java/com/ericsson/bss/rmca/validation...,;,Allow for null or empty entity Allow for null...,;,nan,;,9255
2494,Iffc6857272006f1445fa720df661cd9c681c5aef,;,src/main/java/com/ericsson/bss/rmca/restservic...,;,Update catalog integration message body writer...,;,nan,;,68
2495,Iffca17a9ce002e1504a373167f6320126e5b6e58,;,src/main/java/com/ericsson/bss/rmca/entitymodi...,;,Add expired predicate Add the expired predica...,;,nan,;,9255
2496,Iffd80a804135253ae45cd2f8865491f57dd547c2,;,src/main/java/com/ericsson/bss/rmca/persistenc...,;,Revert Update readAllSharedIds This reverts c...,;,nan,;,68


In [464]:
#convert each line to only containing one file

def create_one_file_lines(commitDF):
    one_line = []
    for row in commitDF.itertuples(index = False):
        pr = row[0]
        files = row[2]
        title = row[4]
        body = row[6]
        author = row[8]
        
        for file in files.split(';'):
            if (not file == ''): #I think somewhere one of the methods is adding a ; at the end
                                 #but this is easier than hunting that problem down
                one_line.append([pr, ';', file, ';', title, ';', body, ';', author])
    
    return pd.DataFrame(one_line)

In [465]:
out_df = create_one_file_lines(PBT_df)

In [466]:
out_df

,0,1,2,3,4,5,6,7,8
0,I001768f8d53c879f8c826e98d24b0130eef344f1,;,src/main/java/com/ericsson/bss/rmca/entitymodi...,;,Validate condition and action spec references,;,nan,;,2591
1,I001768f8d53c879f8c826e98d24b0130eef344f1,;,src/main/java/com/ericsson/bss/rmca/entitymodi...,;,Validate condition and action spec references,;,nan,;,2591
2,I001768f8d53c879f8c826e98d24b0130eef344f1,;,src/main/java/com/ericsson/bss/rmca/validation...,;,Validate condition and action spec references,;,nan,;,2591
3,I001768f8d53c879f8c826e98d24b0130eef344f1,;,src/main/java/com/ericsson/bss/rmca/validation...,;,Validate condition and action spec references,;,nan,;,2591
4,I001768f8d53c879f8c826e98d24b0130eef344f1,;,src/main/java/com/ericsson/bss/rmca/validation...,;,Validate condition and action spec references,;,nan,;,2591
...,...,...,...,...,...,...,...,...,...
33115,Iffca17a9ce002e1504a373167f6320126e5b6e58,;,src/test/java/com/ericsson/bss/rmca/restservic...,;,Add expired predicate Add the expired predica...,;,nan,;,9255
33116,Iffd80a804135253ae45cd2f8865491f57dd547c2,;,src/main/java/com/ericsson/bss/rmca/persistenc...,;,Revert Update readAllSharedIds This reverts c...,;,nan,;,68
33117,Iffd80a804135253ae45cd2f8865491f57dd547c2,;,src/test/java/com/ericsson/bss/rmca/persistenc...,;,Revert Update readAllSharedIds This reverts c...,;,nan,;,68
33118,Ifff275a1012ba92bbbb00e6214c64d6f9fbc30a1,;,src/main/java/com/ericsson/bss/rmca/oamservice...,;,Fix target drop Fix target drop Tracking-Id:...,;,nan,;,5080


In [467]:
out_df.to_csv('fixed_filesPR3BodyTitle2rmca.csv', encoding='utf-8', header=False, index=False, sep='\t')

In [468]:
import numpy as np
with open('fixed_filesPR3BodyTitle2rmca.txt', 'w', encoding='utf-8') as f:
    np.savetxt(f, out_df.values, fmt='%s', delimiter=' ')